# RAG Document Assistant — Day 2: Vector Database & Retrieval
### Storing embeddings permanently and writing the actual "R" (Retrieval) in RAG.

**Stack:** Google Gemini 2.5 Flash + Gemini Embeddings + LangChain + **ChromaDB**

**Recap of Day 1:** we loaded PDFs into `Document` objects, split them into overlapping chunks, and embedded a single test chunk just to see what a vector looks like. Nothing was stored — every rerun re-embedded from scratch.

**Today's goal:** embed *all* chunks, store them permanently in a vector database on disk, and write a function that takes a plain-English question and returns the most relevant chunks. By the end of today, you can ask a question and get back real matching text from your own documents — no LLM answer generation yet (that's Day 3), just retrieval.

---

## Step 1 — Install Dependencies (same as Day 1)

In [1]:
!pip install -q langchain langchain-google-genai langchain-community langchain-text-splitters langchain-chroma chromadb pypdf python-dotenv


[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## Step 2 — Setup API Key

In [2]:
from dotenv import load_dotenv
import os

load_dotenv()

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
assert GEMINI_API_KEY, "GEMINI_API_KEY not found — check your .env file"
print("API key loaded successfully.")

API key loaded successfully.


## Step 3 — Imports

In [3]:
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma

C:\Users\LENOVO\AppData\Local\Temp\ipykernel_10340\852208768.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader


## Step 4 — Load & Split (recap of Day 1, now for ALL your documents)

Same logic as yesterday, just running it on your full `data/` folder instead of one test sentence.

In [4]:
DATA_PATH = "data"
PERSIST_DIR = "chroma_db"

loader = DirectoryLoader(DATA_PATH, glob="**/*.pdf", loader_cls=PyPDFLoader)
documents = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)

print(f"Loaded {len(documents)} page(s), split into {len(chunks)} chunk(s)")

Loaded 3 page(s), split into 13 chunk(s)


## Step 5 — Build & Persist the Vector Store

This is the new piece today. `Chroma.from_documents(...)` does three things in one call:
1. Runs every chunk through the embedding model
2. Stores each chunk's text + metadata + vector inside a Chroma database
3. **Persists it to disk** at `persist_directory="chroma_db"` — so next time you open this notebook, you don't need to re-embed everything from scratch (embedding calls cost time and, on paid tiers, money)

Think of Chroma as a specialized database whose only trick is: *given a query vector, instantly find the stored vectors closest to it.*

In [8]:
embeddings_model = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=os.getenv("GEMINI_API_KEY")
)

vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings_model,
    persist_directory=PERSIST_DIR
)

print(f"Vector store created and persisted to '{PERSIST_DIR}/'")
print(f"Total chunks stored: {vector_store._collection.count()}")

Vector store created and persisted to 'chroma_db/'
Total chunks stored: 13


## Step 6 — Reloading an Existing Store (run this instead of Step 5 next time)

Once `chroma_db/` exists on disk, you don't need to reload PDFs or re-embed anything to use it again — just point Chroma at the existing folder. This cell is commented out today since we just built it fresh above, but this is the pattern you'll use on Day 3 onward.

In [9]:
# Uncomment and run this on future days instead of rebuilding from scratch:

# vector_store = Chroma(
#     persist_directory=PERSIST_DIR,
#     embedding_function=embeddings_model
# )
# print(f"Reloaded existing store with {vector_store._collection.count()} chunks")

## Step 7 — Write the Retrieval Function

This is the actual **"R"** in RAG. `similarity_search(query, k=...)`:
1. Embeds your query text into a vector (same embedding model, so it lands in the same vector space as your chunks)
2. Compares it against every stored chunk vector
3. Returns the **top-k closest matches** — the chunks most likely to contain the answer

No LLM is involved in this step at all — this is pure vector math, which is what makes retrieval fast even over thousands of chunks.

In [10]:
def retrieve(query: str, k: int = 3):
    results = vector_store.similarity_search(query, k=k)
    print(f'Query: "{query}"')
    print(f'Top {k} matching chunk(s):\n')
    for i, doc in enumerate(results, 1):
        source = doc.metadata.get("source", "unknown")
        page = doc.metadata.get("page", "?")
        print(f"--- Result {i} (source: {source}, page: {page}) ---")
        print(doc.page_content[:400])
        print()
    return results

# Try it with a question relevant to whatever you put in data/
test_results = retrieve("What is the main topic of this document?", k=3)

Query: "What is the main topic of this document?"
Top 3 matching chunk(s):

--- Result 1 (source: data\Untitled document.pdf, page: 1) ---
------------------------------------------------------------   Cloud  Computing  and  Modern  Software  Development   Cloud  computing  has  transformed  how  software  applications  are  developed,  deployed,  and  
maintained.
 
Instead
 
of
 
purchasing
 
expensive
 
physical
 
servers,
 
organizations
 
can
 
rent
 
computing
 
resources
 
from
 
cloud
 
providers
 
on
 
demand.
 
This
 
appro

--- Result 2 (source: data\Untitled document.pdf, page: 0) ---
Artificial  Intelligence  and  Its  Impact  on  Modern  Society   Artificial  Intelligence  (AI)  has  become  one  of  the  most  influential  technologies  of  the  21st  
century.
 
It
 
is
 
changing
 
the
 
way
 
businesses
 
operate,
 
how
 
people
 
communicate,
 
and
 
how
 
decisions
 
are
 
made.
 
AI
 
refers
 
to
 
the
 
ability
 
of
 
machines
 
to
 
perform
 
tasks
 
that
 
typic

## Step 8 — Retrieval With Similarity Scores

Sometimes you want to *see* how confident the match is, not just get the top-k blindly. `similarity_search_with_score` returns a **distance score** alongside each chunk — for Chroma's default settings, **lower = more similar** (it's a distance, not a similarity percentage).

This becomes useful later for filtering out weak matches (e.g. "if nothing scores below 0.5, tell the user we don't have relevant info" — prevents the model from making things up when your documents don't actually cover the question).

In [11]:
results_with_scores = vector_store.similarity_search_with_score("What is the main topic of this document?", k=3)

for doc, score in results_with_scores:
    print(f"Score: {score:.4f} | Source: {doc.metadata.get('source', 'unknown')}")
    print(doc.page_content[:200])
    print()

Score: 0.6245 | Source: data\Untitled document.pdf
------------------------------------------------------------   Cloud  Computing  and  Modern  Software  Development   Cloud  computing  has  transformed  how  software  applications  are  developed,  

Score: 0.6315 | Source: data\Untitled document.pdf
Artificial  Intelligence  and  Its  Impact  on  Modern  Society   Artificial  Intelligence  (AI)  has  become  one  of  the  most  influential  technologies  of  the  21st  
century.
 
It
 
is
 
chang

Score: 0.6620 | Source: data\Untitled document.pdf
triggered
 
by
 
events,
 
and
 
organizations
 
only
 
pay
 
for
 
actual
 
execution
 
time.
 
This
 
model
 
is
 
particularly
 
useful
 
for
 
applications
 
with
 
unpredictable
 
workloads.
  As



## Step 9 — The `retriever` Object (what we'll plug into the RAG chain on Day 3)

LangChain has a standard wrapper around a vector store called a **retriever**. It's the same idea as `similarity_search`, just packaged in a shape that plugs directly into chains later — so tomorrow's RAG chain won't need any new retrieval logic, just this object.

In [12]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

# Quick test — same thing as Step 7, different interface
retrieved_docs = retriever.invoke("What is the main topic of this document?")
print(f"Retriever returned {len(retrieved_docs)} chunk(s)")

Retriever returned 3 chunk(s)


---
## Day 2 Wrap-Up

Today you completed the full **ingestion + retrieval** side of RAG:

`Load → Split → Embed → Store (persisted) → Retrieve top-k chunks for a query`

You now have a `chroma_db/` folder on disk holding real, searchable embeddings of your documents, and a working `retrieve()` function plus a `retriever` object.

**What's still missing:** the model never actually *answers* anything yet — we're only fetching raw chunks. **Tomorrow (Day 3)** we take these retrieved chunks, stuff them into a prompt alongside the user's question, send that to Gemini, and get a real grounded answer — the full RAG chain, end to end.

See `README_Day2.md` for the full write-up of today's concepts.